# Tugas Besar Part 3 - Training Model PyTorch dan Ekspor `.pth`

Notebook ini melatih model **MLP (PyTorch)** untuk klasifikasi **Breast Cancer Wisconsin** (0 = malignant, 1 = benign),
mengevaluasinya, lalu menyimpan model ke file **`breast_cancer_model.pth`** yang akan dipakai aplikasi web (Flask).

**Cara pakai di Google Colab:** jalankan semua sel (Runtime > Run all). Sel terakhir otomatis mengunduh file `.pth`.
Ambil screenshot saat proses training selesai dan saat file `.pth` terunduh untuk laporan.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("PyTorch:", torch.__version__)

## 1. Preprocessing

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = data.target.astype(int)      # 0 = malignant, 1 = benign
print("Ukuran data:", X.shape, "| missing value:", int(X.isnull().sum().sum()))

# Split 80:20 stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

# Buang fitur berkorelasi sangat tinggi (> 0.95); dihitung dari data latih saja (tanpa data leakage)
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.95).any()]
X_train = X_train.drop(columns=to_drop)
X_test = X_test.drop(columns=to_drop)
feature_names = list(X_train.columns)
print(f"Fitur dibuang ({len(to_drop)}):", to_drop)
print("Fitur dipakai:", len(feature_names))

# Validation split dari data latih (untuk early stopping)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=SEED)

# Standardisasi: mean dan std dihitung dari X_tr saja
mean = X_tr.mean().values.astype("float32")
scale = X_tr.std(ddof=0).values.astype("float32")


def to_x(df):
    return torch.tensor((df.values - mean) / scale, dtype=torch.float32)


def to_y(s):
    return torch.tensor(s.values, dtype=torch.float32)


Xtr_t, Xval_t, Xte_t, Xfull_t = to_x(X_tr), to_x(X_val), to_x(X_test), to_x(X_train)
ytr_t, yval_t, yte_t = to_y(y_tr), to_y(y_val), to_y(y_test)
print("Train:", tuple(Xtr_t.shape), "| Val:", tuple(Xval_t.shape), "| Test:", tuple(Xte_t.shape))

## 2. Arsitektur Model

Kelas ini **sama persis** dengan `model.py` pada aplikasi web, karena file `.pth` hanya menyimpan bobot.

In [ ]:
class BreastCancerMLP(nn.Module):
    """MLP untuk klasifikasi biner. Keluaran berupa logit (belum sigmoid)."""

    def __init__(self, in_features, hidden=(32, 16), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


HIDDEN, DROPOUT = (32, 16), 0.2
model = BreastCancerMLP(len(feature_names), HIDDEN, DROPOUT)
print(model)
print("Jumlah parameter:", sum(p.numel() for p in model.parameters()))

## 3. Training dengan Early Stopping

In [ ]:
train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

MAX_EPOCHS, PATIENCE = 300, 25
best_val, best_state, wait = float("inf"), None, 0
hist = {"train": [], "val": []}

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        tr_loss = criterion(model(Xtr_t), ytr_t).item()
        val_loss = criterion(model(Xval_t), yval_t).item()
    hist["train"].append(tr_loss); hist["val"].append(val_loss)

    if val_loss < best_val - 1e-4:
        best_val, wait = val_loss, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        wait += 1
    if epoch % 20 == 0 or epoch == 1:
        print(f"epoch {epoch:3d} | train loss {tr_loss:.4f} | val loss {val_loss:.4f}")
    if wait >= PATIENCE:
        print(f"Early stopping pada epoch {epoch} (val loss terbaik {best_val:.4f})")
        break

model.load_state_dict(best_state)
model.eval()

plt.figure(figsize=(6, 3.6))
plt.plot(hist["train"], label="Train loss"); plt.plot(hist["val"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.title("Kurva Loss"); plt.legend(); plt.tight_layout(); plt.show()

## 4. Evaluasi dan Analisis Goodfit

In [ ]:
def predict_proba(m, X_t):
    m.eval()
    with torch.no_grad():
        return torch.sigmoid(m(X_t)).numpy()


p_train = predict_proba(model, Xfull_t)
p_test = predict_proba(model, Xte_t)
pred_train = (p_train >= 0.5).astype(int)
pred_test = (p_test >= 0.5).astype(int)

metrics = {
    "train_accuracy": float(accuracy_score(y_train, pred_train)),
    "test_accuracy": float(accuracy_score(y_test, pred_test)),
    "precision": float(precision_score(y_test, pred_test)),
    "recall": float(recall_score(y_test, pred_test)),
    "f1": float(f1_score(y_test, pred_test)),
    "roc_auc": float(roc_auc_score(y_test, p_test)),
}
for k, v in metrics.items():
    print(f"{k:15s}: {v:.4f}")
print("\nConfusion matrix (baris = aktual, kolom = prediksi; urutan: malignant, benign):")
print(confusion_matrix(y_test, pred_test))

gap = metrics["train_accuracy"] - metrics["test_accuracy"]
if metrics["train_accuracy"] < 0.90 and metrics["test_accuracy"] < 0.90:
    status = "Underfit"
elif gap > 0.03:
    status = "Overfit"
else:
    status = "Goodfit"
print(f"\nSelisih akurasi train - test = {gap:.4f} -> {status}")

## 5. Simpan Model ke `.pth`

Yang disimpan: bobot model, konfigurasi arsitektur, nama fitur, parameter standardisasi, rentang fitur pada data latih,
contoh data (satu benign, satu malignant yang diprediksi benar), dan metrik. Aplikasi web hanya butuh file ini.

In [ ]:
MODEL_PATH = "breast_cancer_model.pth"


def pick_example(label):
    """Ambil satu sampel uji berkelas `label` yang diprediksi benar oleh model."""
    idx = np.where((y_test.values == label) & (pred_test == label))[0][0]
    return {name: float(X_test.iloc[idx][name]) for name in feature_names}


checkpoint = {
    "state_dict": model.state_dict(),
    "in_features": len(feature_names),
    "hidden": list(HIDDEN),
    "dropout": DROPOUT,
    "feature_names": feature_names,
    "mean": [float(v) for v in mean],
    "scale": [float(v) for v in scale],
    "feature_min": [float(v) for v in X_train.min().values],
    "feature_max": [float(v) for v in X_train.max().values],
    "class_names": ["Malignant (ganas)", "Benign (jinak)"],     # indeks = nilai target
    "examples": {"benign": pick_example(1), "malignant": pick_example(0)},
    "metrics": metrics,
}
torch.save(checkpoint, MODEL_PATH)

import os
print(f"Tersimpan: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024:.1f} KB)")

## 6. Verifikasi: muat ulang `.pth` seperti yang dilakukan aplikasi

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
reloaded = BreastCancerMLP(ckpt["in_features"], tuple(ckpt["hidden"]), ckpt["dropout"])
reloaded.load_state_dict(ckpt["state_dict"])
reloaded.eval()

p_reloaded = predict_proba(reloaded, Xte_t)
assert np.allclose(p_reloaded, p_test, atol=1e-6), "Hasil prediksi setelah dimuat ulang berbeda!"
print("OK: model yang dimuat ulang memberi prediksi identik pada", len(p_test), "data uji.")
print("Fitur:", len(ckpt["feature_names"]), "| Metrik tersimpan:", {k: round(v, 4) for k, v in ckpt["metrics"].items()})

## 7. Unduh file `.pth`

Jalankan sel ini di Google Colab; browser akan mengunduh `breast_cancer_model.pth`.
Letakkan file tersebut di folder `model/` pada proyek aplikasi.

In [ ]:
try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    print("Bukan di Google Colab. File tersimpan di:", os.path.abspath(MODEL_PATH))